# Generacion con el modelo RAFT local (Ollama), 2 + 1 estilos de prompt

Este notebook es la extension de `generacion_llm.ipynb` para el modelo ajustado con RAFT (Llama-3.1-8B-Instruct + LoRA, cuantizado a GGUF Q4_K_M y servido con **Ollama** en la laptop). **No modifica nada existente**: escribe sus propios checkpoints y su propio archivo de resultados.

**Mismo diseno experimental que las 8 combinaciones anteriores:**
- Se usan las mismas **196 preguntas**, con el mismo filtro (`llm_context_precision_with_reference > 0` y `context_recall > 0` en ambos pipelines).
- Se usan solo los **contextos ya recuperados por `openai_cs800`** (OpenAI `text-embedding-3-large` denso, cs800, k=5), la mejor configuracion de retrieval y la del pipeline final. Aqui no se vuelve a hacer retrieval.
- `temperature=0` y reintentos sobre el mismo modelo, sin fallback.
- La exportacion tiene el **mismo formato** de `resultados_generacion_llm.json` (`combinaciones` → `resultados` con `respuesta_generada`, `contextos_usados`, etc.), asi que `evaluacion_generacion_ragas.ipynb` lo puede leer sin cambios en su logica.

**Estilos de prompt:**
- `directo` y `estructurado`: exactamente los mismos textos de `generacion_llm.ipynb`, para comparar 1 a 1 con Sonnet 5 y Haiku 4.5.
- `raft`: el formato con el que se **entreno** el modelo (system prompt + `<DOCUMENTO>` + `##Reason / ##begin_quote## / ##Answer`). RAFT supone que el modelo ve en prueba el mismo formato que en entrenamiento (Zhang et al., 2024, Sec. 3). Por eso este estilo mide el metodo tal como se propone, y `directo`/`estructurado` miden que tan bien se generaliza a otros prompts.

1 pipeline × 1 modelo × 3 prompts = **3 combinaciones**. Se comparan contra las 4 combinaciones de `openai_cs800` del notebook original (Sonnet 5 y Haiku 4.5 × directo/estructurado). El modelo base sin ajustar (control equivalente a "LLaMA2-7B + RAG" del paper) se puede activar en la configuracion.

**Salidas:**
- `./validacion/checkpoints_llm/openai_cs800__raft-llama3.1-8b__<prompt>.json`: archivos nuevos, con la misma convencion de nombres.
- `./validacion/resultados_generacion_llm_raft.json`: mismo formato que `resultados_generacion_llm.json`.
- (opcional, ultima celda) `./validacion/resultados_generacion_llm_con_raft.json`: las 8 combinaciones anteriores mas las nuevas, para evaluarlas juntas con RAGAS.

## 0. Preparar el modelo en Ollama (una sola vez)

1. Instala **Ollama para Windows** desde https://ollama.com/download y dejalo corriendo (icono en la bandeja del sistema).
2. **Opcional, pero recomendado con 6 GB de VRAM:** crea estas dos variables de entorno de Windows y reinicia Ollama. Reducen a la mitad la memoria del contexto, para que quepa mas del modelo en la GPU.
   - `OLLAMA_FLASH_ATTENTION` = `1`
   - `OLLAMA_KV_CACHE_TYPE` = `q8_0`
3. Consigue los dos archivos del modelo: `raft-acreditacion-8b-Q4_K_M.gguf` y `Modelfile`.
   - **Desde Kaggle (lo mas seguro):** abre la version terminada del notebook de entrenamiento, ve a la pestana **Output** y descarga la carpeta `gguf/`.
   - **Desde Hugging Face:** repositorio `<usuario>/raft-acreditacion-8b-gguf` → *Files and versions*. Si la pagina del repositorio solo muestra un ejemplo de `upload_folder(...)`, el repositorio esta **vacio** (la subida no se completo) y hay que usar la opcion de Kaggle.
4. Pon los dos archivos en una carpeta, por ejemplo `.\modelos\raft\`, y en una terminal dentro de esa carpeta ejecuta:
   ```
   ollama create raft-acreditacion -f Modelfile
   ```
5. Verifica con `ollama list` que aparece `raft-acreditacion`.

> El GGUF pesa ~5 GB. Si la carpeta `modelos/` queda dentro del proyecto, agregala a `.gitignore`.

In [ ]:
%pip install -q pandas httpx

## 1. Configuracion

In [15]:
import json
import re
import time
from pathlib import Path

import httpx
import pandas as pd

OLLAMA_URL = "http://localhost:11434"

# Modelos locales a evaluar. La clave es el nombre que queda en "modelo_llm" (y en los nombres de archivo).
MODELOS_LOCALES = {
    "raft-llama3.1-8b": {
        "ollama_id": "raft-acreditacion",   # nombre usado en `ollama create`
        "rol": "raft",
    },
    # Control opcional: el MISMO modelo base sin ajustar, con la misma cuantizacion.
    # Para activarlo: `ollama pull llama3.1:8b-instruct-q4_K_M` y descomenta.
    # "llama3.1-8b-base": {
    #     "ollama_id": "llama3.1:8b-instruct-q4_K_M",
    #     "rol": "base_sin_ajustar",
    # },
}

# Orden de ejecucion: primero el formato nativo de RAFT, que es el resultado principal
ESTILOS_A_CORRER = ["raft", "estructurado", "directo"]

# Los prompts de este flujo (5 fragmentos de ~800 tokens) llegan a ~3-6k tokens; 8192 cubre todos con margen.
# Ollama RECORTA EN SILENCIO lo que exceda num_ctx, por eso se verifica en cada respuesta.
NUM_CTX = 8192
MAX_TOKENS_RESPUESTA = 1024
SEMILLA = 42

# None = las 196 preguntas filtradas. Pon un numero chico (ej. 5) para una prueba rapida.
# OJO: si corres con muestra, borra los checkpoints *__raft-llama3.1-8b__* antes de la corrida completa.
TAMANO_MUESTRA_LLM = None

RUTA_CHECKPOINTS_LLM = Path("./validacion/checkpoints_llm")
RUTA_CHECKPOINTS_LLM.mkdir(parents=True, exist_ok=True)
RUTA_SALIDA_LLM = "./validacion/resultados_generacion_llm_raft.json"

TAMANO_MUESTRA_LLM = None

## 2. Verificar que Ollama y el modelo estan disponibles

In [16]:
_http = httpx.Client(base_url=OLLAMA_URL, timeout=httpx.Timeout(900.0, connect=10.0))

try:
    modelos_instalados = [m["name"] for m in _http.get("/api/tags").json()["models"]]
except Exception as e:
    raise RuntimeError(f"No pude conectar con Ollama en {OLLAMA_URL}. ¿Esta abierto? ({e})")

print("Modelos en Ollama:", modelos_instalados)
for nombre, info in MODELOS_LOCALES.items():
    ok = any(m == info["ollama_id"] or m.split(":")[0] == info["ollama_id"] for m in modelos_instalados)
    print(f"  {nombre:22s} -> {info['ollama_id']:32s} {'OK' if ok else 'NO ENCONTRADO'}")
    if not ok:
        raise RuntimeError(f"Falta el modelo '{info['ollama_id']}' en Ollama (ver seccion 0).")

Modelos en Ollama: ['raft-acreditacion:latest']
  raft-llama3.1-8b       -> raft-acreditacion                OK


## 3. Mismas preguntas y mismos contextos que `generacion_llm.ipynb`

El filtro es identico al original: la interseccion de preguntas con exito en ambos pipelines, 196 preguntas. Asi las respuestas son comparables 1 a 1 con las combinaciones `openai_cs800` ya generadas. Para generar solo se usan los contextos de `openai_cs800`.

In [17]:
RUTA_EVALUACION_DETALLE = "./validacion/evaluacion_ragas_detalle.csv"
RUTA_RETRIEVAL_OPENAI = "./validacion/resultados_retrieval.json"

ETIQUETA_OPENAI = "openai/text-embedding-3-large_cs800"
ETIQUETA_BGE = "BAAI/bge-m3_cs800_hibrido"

evaluacion = pd.read_csv(RUTA_EVALUACION_DETALLE)


def preguntas_con_score_positivo(etiqueta):
    sub = evaluacion[evaluacion["configuracion"] == etiqueta]
    validas = sub[(sub["llm_context_precision_with_reference"] > 0) & (sub["context_recall"] > 0)]
    return set(validas["user_input"])


preguntas_validas = preguntas_con_score_positivo(ETIQUETA_OPENAI) & preguntas_con_score_positivo(ETIQUETA_BGE)


def cargar_resultados_config(ruta, etiqueta_config=None):
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = json.load(f)
    if "configuraciones" in contenido:
        for c in contenido["configuraciones"]:
            if f"{c['modelo_embedding']}_cs{c['chunk_size']}" == etiqueta_config:
                return c["resultados"]
        raise ValueError(f"No se encontro la configuracion '{etiqueta_config}' en {ruta}")
    return contenido["resultados"]


resultados_openai = sorted(
    (r for r in cargar_resultados_config(RUTA_RETRIEVAL_OPENAI, ETIQUETA_OPENAI) if r["pregunta"] in preguntas_validas),
    key=lambda r: r["pregunta"],
)

# Solo la mejor configuracion de retrieval
PIPELINES_RETRIEVE = {
    "openai_cs800": resultados_openai,
}
print(f"{len(resultados_openai)} preguntas (deben ser 196, igual que en generacion_llm.ipynb)")

196 preguntas (deben ser 196, igual que en generacion_llm.ipynb)


## 4. Estilos de prompt

`directo` y `estructurado` son copia textual de `generacion_llm.ipynb`. `raft` es copia de `raft_formato.py` (proyecto de corpus). Si se cambia ese archivo, hay que cambiarlo aqui tambien, porque debe ser **identico** al formato de entrenamiento.

In [18]:
def formatear_contexto(contextos):
    return "\n\n".join(f"--- Fragmento {i + 1} ---\n{c}" for i, c in enumerate(contextos))


def prompt_directo(pregunta, contextos):
    return f'''Responde la siguiente pregunta usando UNICAMENTE la informacion del contexto. Si el contexto no contiene la respuesta, dilo explicitamente en vez de inventar o completar con conocimiento externo.

Contexto:
{formatear_contexto(contextos)}

Pregunta: {pregunta}

Respuesta:'''


def prompt_estructurado(pregunta, contextos):
    return f'''Eres un asistente que responde preguntas basandote ESTRICTAMENTE en el contexto proporcionado, sin usar conocimiento externo. Sigue estos pasos antes de responder:

1. Identifica que fragmentos del contexto son relevantes para la pregunta.
2. Verifica si, con esos fragmentos, la pregunta se puede responder de forma completa, parcial, o no se puede responder.
3. Redacta la respuesta final basandote unicamente en lo que identificaste en el paso 1.

Contexto:
{formatear_contexto(contextos)}

Pregunta: {pregunta}

Responde siguiendo EXACTAMENTE este formato:
Fragmentos relevantes: <cuales y por que, brevemente>
Cobertura: <completa / parcial / nula>
Respuesta final: <tu respuesta aqui; si la cobertura es nula, escribe "No se encuentra en el contexto">'''


# --- formato RAFT (copia de raft_formato.py) ---
RAFT_MARCA_RESPUESTA = "##Answer:"
RAFT_SYSTEM_PROMPT = (
    "Eres un asistente que responde preguntas sobre el informe de autoevaluación "
    "con fines de acreditación del programa de Ingeniería de Sistemas. "
    "Recibirás varios documentos recuperados; algunos pueden no ser relevantes para la pregunta. "
    "Usa solo la información pertinente: razona paso a paso, copia textualmente los fragmentos "
    "que sustentan tu respuesta entre ##begin_quote## y ##end_quote##, y termina con la respuesta "
    "final después de '##Answer:'. Responde siempre en español."
)


def mensajes_raft(pregunta, contextos):
    documentos = "\n\n".join(f"<DOCUMENTO>\n{d.strip()}\n</DOCUMENTO>" for d in contextos)
    return [
        {"role": "system", "content": RAFT_SYSTEM_PROMPT},
        {"role": "user", "content": f"{documentos}\n\nPregunta: {pregunta.strip()}"},
    ]


# Cada estilo devuelve la lista de mensajes. directo/estructurado: un solo mensaje "user", como en el original.
ESTILOS_PROMPT = {
    "raft": mensajes_raft,
    "estructurado": lambda p, c: [{"role": "user", "content": prompt_estructurado(p, c)}],
    "directo": lambda p, c: [{"role": "user", "content": prompt_directo(p, c)}],
}


def extraer_respuesta_final(texto_generado, nombre_prompt):
    if texto_generado is None:
        return None
    if nombre_prompt == "estructurado":
        m = re.search(r"Respuesta final:\s*(.*)", texto_generado, re.DOTALL)
        return m.group(1).strip() if m else texto_generado.strip()
    if nombre_prompt == "raft":
        if RAFT_MARCA_RESPUESTA in texto_generado:
            return texto_generado.split(RAFT_MARCA_RESPUESTA, 1)[1].strip()
        return texto_generado.strip()
    return texto_generado.strip()

## 5. Cliente Ollama con reintentos (sin fallback de modelo)

Se usa la API nativa de Ollama (`/api/chat`) en lugar de la compatible con OpenAI por dos razones:
- permite fijar `num_ctx` en cada llamada;
- devuelve los conteos de tokens y los tiempos de cada etapa, utiles para el analisis de latencias.

In [19]:
def llamar_ollama(ollama_id, mensajes, intentos=5, espera_inicial=5):
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            t0 = time.perf_counter()
            r = _http.post("/api/chat", json={
                "model": ollama_id,
                "messages": mensajes,
                "stream": False,
                "options": {
                    "temperature": 0,
                    "seed": SEMILLA,
                    "num_ctx": NUM_CTX,
                    "num_predict": MAX_TOKENS_RESPUESTA,
                },
            })
            r.raise_for_status()
            d = r.json()
            tokens_prompt = d.get("prompt_eval_count") or 0
            tokens_generados = d.get("eval_count") or 0
            return {
                "texto": d["message"]["content"],
                "latencia_s": round(time.perf_counter() - t0, 3),
                "latencia_prompt_s": round((d.get("prompt_eval_duration") or 0) / 1e9, 3),
                "latencia_generacion_s": round((d.get("eval_duration") or 0) / 1e9, 3),
                "tokens_prompt": tokens_prompt,
                "tokens_generados": tokens_generados,
                # Si el prompt llena el contexto, Ollama pudo haber recortado el inicio
                "contexto_posiblemente_truncado": tokens_prompt >= NUM_CTX - 8,
                "respuesta_cortada_por_longitud": d.get("done_reason") == "length",
            }
        except Exception as e:
            ultimo_error = e
            espera = espera_inicial * (2 ** (intento - 1))
            print(f"    intento {intento}/{intentos} fallo ({e}); esperando {espera}s...")
            time.sleep(espera)
    print(f"    se agotaron los reintentos para {ollama_id}: {ultimo_error}")
    return None

## 6. Ejemplo rapido y estimacion de tiempo

La primera llamada carga el modelo en la GPU y tarda mas. Despues de esta celda, ejecuta `ollama ps` en una terminal: la columna **PROCESSOR** dice que porcentaje quedo en GPU. Con `100% GPU` es lo ideal. Con un reparto CPU/GPU funciona, pero mas lento.

In [20]:
nombre_modelo, info = next(iter(MODELOS_LOCALES.items()))
ejemplo = resultados_openai[0]
print("Pregunta:", ejemplo["pregunta"])
print("Esperada:", ejemplo.get("respuesta_esperada"), "\n")

_ = llamar_ollama(info["ollama_id"], [{"role": "user", "content": "Hola"}])   # calentamiento (carga del modelo)

tiempos = []
for nombre_prompt in ESTILOS_A_CORRER:
    res = llamar_ollama(info["ollama_id"], ESTILOS_PROMPT[nombre_prompt](ejemplo["pregunta"], ejemplo["contextos_recuperados"]))
    tiempos.append(res["latencia_s"])
    print(f"=== {nombre_prompt}  ({res['latencia_s']} s, prompt={res['tokens_prompt']} tok, salida={res['tokens_generados']} tok)")
    print(res["texto"][:1200])
    print(f"--> respuesta_generada: {extraer_respuesta_final(res['texto'], nombre_prompt)[:300]}\n")

n_comb = len(PIPELINES_RETRIEVE) * len(MODELOS_LOCALES) * len(ESTILOS_A_CORRER)
n_preg = TAMANO_MUESTRA_LLM or len(resultados_openai)
print(f"Estimacion corrida completa: ~{sum(tiempos)/len(tiempos) * n_comb * n_preg / 3600:.1f} h "
      f"({n_comb} combinaciones x {n_preg} preguntas)")

Pregunta: Compare la duración en semestres y el total de créditos de los programas de Ingeniería de Sistemas de la Universidad de Antioquia y la Universidad Nacional de Colombia según la Tabla 1.3.
Esperada: La Universidad de Antioquia tiene un programa de Ingeniería de Sistemas con duración de 10 semestres y 180 créditos totales, mientras que la Universidad Nacional de Colombia tiene un programa con duración de 10 semestres y 165 créditos totales. 

=== raft  (14.662 s, prompt=2504 tok, salida=220 tok)
##Reason: La Tabla 1.3 muestra la información de los programas de Ingeniería de Sistemas de diferentes universidades, incluyendo la Universidad de Antioquia y la Universidad Nacional de Colombia. Para la Universidad de Antioquia, se indica ##begin_quote##|Universidad de<br>Antioquia|Medellín|Ingeniería de sistemas|Presencial -<br>Diurna/ Virtual|10|180|##end_quote## y para la Universidad Nacional de Colombia, se indica ##begin_quote##|Universidad Nacional de<br>Colombia|Bogotá|Ingenierí

## 7. Generacion completa (3 combinaciones, con checkpoint)

Funciona como el notebook original: cada combinacion se guarda en su checkpoint al terminar, y si existe se reutiliza. Como el modelo local es mas lento, ademas se guarda un **checkpoint parcial cada 10 preguntas** (`*.parcial.json`). Si la laptop se suspende o se interrumpe la celda, al volver a ejecutarla continua donde iba.

Para que no se interrumpa: conecta la laptop a la corriente y desactiva la suspension mientras corre.

In [21]:
def generar_para_combinacion(nombre_pipeline, resultados_pipeline, nombre_modelo, ollama_id, nombre_prompt, funcion_prompt):
    etiqueta = f"{nombre_pipeline}__{nombre_modelo}__{nombre_prompt}"
    ruta_checkpoint = RUTA_CHECKPOINTS_LLM / f"{etiqueta}.json"
    ruta_parcial = RUTA_CHECKPOINTS_LLM / f"{etiqueta}.parcial.json"

    if ruta_checkpoint.exists():
        print(f"  [{etiqueta}] ya generado antes, se reutiliza el checkpoint.")
        with open(ruta_checkpoint, "r", encoding="utf-8") as f:
            return json.load(f)

    preguntas_a_usar = resultados_pipeline[:TAMANO_MUESTRA_LLM] if TAMANO_MUESTRA_LLM else resultados_pipeline

    salida = []
    if ruta_parcial.exists():
        with open(ruta_parcial, "r", encoding="utf-8") as f:
            salida = json.load(f)
        print(f"  [{etiqueta}] retomando desde la pregunta {len(salida) + 1}")
    hechas = {r["id"] for r in salida}

    for i, r in enumerate(preguntas_a_usar, 1):
        if r["id"] in hechas:
            continue
        res = llamar_ollama(ollama_id, funcion_prompt(r["pregunta"], r["contextos_recuperados"]))
        respuesta_cruda = res["texto"] if res else None
        salida.append({
            "id": r["id"],
            "pregunta": r["pregunta"],
            "respuesta_esperada": r.get("respuesta_esperada"),
            "evidencia_gold": r.get("evidencia_gold"),
            "tipo": r.get("tipo"),
            "dificultad": r.get("dificultad"),
            "tipo_evidencia": r.get("tipo_evidencia"),
            "contextos_usados": r["contextos_recuperados"],
            "respuesta_generada_cruda": respuesta_cruda,
            "respuesta_generada": extraer_respuesta_final(respuesta_cruda, nombre_prompt),
            # campos adicionales (no existen en las combinaciones de OpenRouter)
            "latencia_s": res["latencia_s"] if res else None,
            "latencia_prompt_s": res["latencia_prompt_s"] if res else None,
            "latencia_generacion_s": res["latencia_generacion_s"] if res else None,
            "tokens_prompt": res["tokens_prompt"] if res else None,
            "tokens_generados": res["tokens_generados"] if res else None,
            "contexto_posiblemente_truncado": res["contexto_posiblemente_truncado"] if res else None,
            "respuesta_cortada_por_longitud": res["respuesta_cortada_por_longitud"] if res else None,
        })
        if len(salida) % 10 == 0:
            with open(ruta_parcial, "w", encoding="utf-8") as f:
                json.dump(salida, f, ensure_ascii=False)
        if i % 25 == 0 or i == len(preguntas_a_usar):
            print(f"    [{etiqueta}] {i}/{len(preguntas_a_usar)}")

    # mismo orden que el original (por pregunta)
    orden = {r["id"]: k for k, r in enumerate(preguntas_a_usar)}
    salida.sort(key=lambda x: orden.get(x["id"], 10**9))

    with open(ruta_checkpoint, "w", encoding="utf-8") as f:
        json.dump(salida, f, ensure_ascii=False, indent=2)
    if ruta_parcial.exists():
        ruta_parcial.unlink()
    print(f"  [{etiqueta}] listo, {len(salida)} respuestas guardadas en {ruta_checkpoint}")
    return salida

In [23]:
combinaciones_resultado = []

for nombre_pipeline, resultados_pipeline in PIPELINES_RETRIEVE.items():
    for nombre_modelo, info_modelo in MODELOS_LOCALES.items():
        for nombre_prompt in ESTILOS_A_CORRER:
            print(f"\n=== {nombre_pipeline} | {nombre_modelo} | prompt={nombre_prompt} ===")
            resultados_generados = generar_para_combinacion(
                nombre_pipeline, resultados_pipeline, nombre_modelo,
                info_modelo["ollama_id"], nombre_prompt, ESTILOS_PROMPT[nombre_prompt],
            )
            combinaciones_resultado.append({
                "pipeline_retrieve": nombre_pipeline,
                "modelo_llm": nombre_modelo,
                "openrouter_id": f"ollama/{info_modelo['ollama_id']}",   # mismo campo que el original; aqui es local
                "rol_modelo": info_modelo["rol"],
                "estilo_prompt": nombre_prompt,
                "total_preguntas": len(resultados_generados),
                "resultados": resultados_generados,
            })

salida_final = {
    "dataset_fuente": "dataset_validacion_final.json",
    "criterio_filtro": "preguntas con llm_context_precision_with_reference > 0 y context_recall > 0 en AMBOS pipelines de retrieve (openai cs800 denso, BGE-m3 hibrido cs800)",
    "total_preguntas_filtradas": len(resultados_openai),
    "generacion": {
        "motor": "ollama (local)",
        "modelo": "Llama-3.1-8B-Instruct + LoRA RAFT, GGUF Q4_K_M",
        "num_ctx": NUM_CTX, "max_tokens": MAX_TOKENS_RESPUESTA, "temperature": 0, "seed": SEMILLA,
        "contextos": "solo openai_cs800 (text-embedding-3-large denso, cs800, k=5)",
    },
    "combinaciones": combinaciones_resultado,
}

with open(RUTA_SALIDA_LLM, "w", encoding="utf-8") as f:
    json.dump(salida_final, f, ensure_ascii=False, indent=2)
print(f"\nListo. {len(combinaciones_resultado)} combinaciones guardadas en {RUTA_SALIDA_LLM}")


=== openai_cs800 | raft-llama3.1-8b | prompt=raft ===
    [openai_cs800__raft-llama3.1-8b__raft] 25/196
    [openai_cs800__raft-llama3.1-8b__raft] 50/196
    [openai_cs800__raft-llama3.1-8b__raft] 75/196
    [openai_cs800__raft-llama3.1-8b__raft] 100/196
    [openai_cs800__raft-llama3.1-8b__raft] 125/196
    [openai_cs800__raft-llama3.1-8b__raft] 150/196
    [openai_cs800__raft-llama3.1-8b__raft] 175/196
    [openai_cs800__raft-llama3.1-8b__raft] 196/196
  [openai_cs800__raft-llama3.1-8b__raft] listo, 196 respuestas guardadas en validacion\checkpoints_llm\openai_cs800__raft-llama3.1-8b__raft.json

=== openai_cs800 | raft-llama3.1-8b | prompt=estructurado ===
    [openai_cs800__raft-llama3.1-8b__estructurado] 25/196
    [openai_cs800__raft-llama3.1-8b__estructurado] 50/196
    [openai_cs800__raft-llama3.1-8b__estructurado] 75/196
    [openai_cs800__raft-llama3.1-8b__estructurado] 100/196
    [openai_cs800__raft-llama3.1-8b__estructurado] 125/196
    [openai_cs800__raft-llama3.1-8b__est

## 8. Verificacion

In [25]:
tamano_mb = Path(RUTA_SALIDA_LLM).stat().st_size / (1024 * 1024)
print(f"Archivo: {RUTA_SALIDA_LLM} ({tamano_mb:.2f} MB)\n")

filas = []
for c in combinaciones_resultado:
    rs = c["resultados"]
    lat = [r["latencia_s"] for r in rs if r.get("latencia_s") is not None]
    filas.append({
        "combinacion": f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}",
        "con_respuesta": sum(1 for r in rs if r["respuesta_generada"]),
        "total": c["total_preguntas"],
        "raft_con_##Answer": sum(1 for r in rs if (r["respuesta_generada_cruda"] or "").count("##Answer:")) if c["estilo_prompt"] == "raft" else None,
        "estructurado_con_formato": sum(1 for r in rs if "Respuesta final:" in (r["respuesta_generada_cruda"] or "")) if c["estilo_prompt"] == "estructurado" else None,
        "contexto_truncado": sum(1 for r in rs if r.get("contexto_posiblemente_truncado")),
        "respuesta_cortada": sum(1 for r in rs if r.get("respuesta_cortada_por_longitud")),
        "latencia_media_s": round(sum(lat) / len(lat), 2) if lat else None,
        "tokens_prompt_medio": round(sum(r["tokens_prompt"] or 0 for r in rs) / max(1, len(rs))),
    })
pd.DataFrame(filas)

Archivo: ./validacion/resultados_generacion_llm_raft.json (7.73 MB)



,combinacion,con_respuesta,total,raft_con_##Answer,estructurado_con_formato,contexto_truncado,respuesta_cortada,latencia_media_s,tokens_prompt_medio
0,openai_cs800__raft-llama3.1-8b__raft,196,196,194.0,NaN,0,2,13.19,3203
1,openai_cs800__raft-llama3.1-8b__estructurado,196,196,NaN,196.0,0,0,11.39,3263
2,openai_cs800__raft-llama3.1-8b__directo,196,196,NaN,NaN,0,1,8.03,3128


**Como leer la tabla:**
- `contexto_truncado` debe ser 0. Si no lo es, sube `NUM_CTX` (por ejemplo a 12288), borra los checkpoints de ese modelo y repite.
- `respuesta_cortada` > 0 indica que el modelo llego a `MAX_TOKENS_RESPUESTA`. En `raft` esas respuestas pueden quedar sin `##Answer:`.
- `raft_con_##Answer` mide si el modelo aprendio el formato. Si es bajo con estos contextos, revisa la nota sobre longitud al final.

## 9. (Opcional) Un solo archivo para evaluar todo junto con RAGAS

Crea `resultados_generacion_llm_con_raft.json`, con las 8 combinaciones originales mas las 3 nuevas. Sigue estos pasos:
1. En `evaluacion_generacion_ragas.ipynb`, cambia solo `RUTA_GENERACION` a este archivo nuevo.
2. Las 8 combinaciones anteriores reutilizan sus checkpoints de evaluacion, asi que no se vuelven a pagar. Solo se evaluan las nuevas.
3. Para no sobrescribir el detalle y el resumen anteriores, cambia tambien `RUTA_SALIDA_DETALLE` y `RUTA_SALIDA_RESUMEN` a nombres nuevos (por ejemplo, con sufijo `_con_raft`).

In [26]:
RUTA_ORIGINAL = "./validacion/resultados_generacion_llm.json"
RUTA_COMBINADA = "./validacion/resultados_generacion_llm_con_raft.json"

with open(RUTA_ORIGINAL, "r", encoding="utf-8") as f:
    original = json.load(f)

etiquetas_previas = {f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}" for c in original["combinaciones"]}
nuevas = [c for c in combinaciones_resultado
          if f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}" not in etiquetas_previas]

combinado = dict(original)
combinado["combinaciones"] = original["combinaciones"] + nuevas
combinado["nota"] = "8 combinaciones OpenRouter (generacion_llm.ipynb) + combinaciones RAFT locales (generacion_llm_raft.ipynb)"

with open(RUTA_COMBINADA, "w", encoding="utf-8") as f:
    json.dump(combinado, f, ensure_ascii=False, indent=2)
print(f"{len(combinado['combinaciones'])} combinaciones en {RUTA_COMBINADA}")

11 combinaciones en ./validacion/resultados_generacion_llm_con_raft.json
